# E0.4 — Random-entry MFE / MAE baseline (intraday, BTCUSDT 5m, 2025)

**Question:** is "74% of my trades reached +1% MFE" evidence of a good entry, or just what BTC does to *any* entry?

**Method:** every 5m close in 2025 is treated as an entry (long and short), so this is the full unconditional distribution, with no sampling noise. For intraday holding periods (30m to 24h) we measure:

1. MFE / MAE distributions (how far price goes for / against you)
2. Which comes first, +TP or −SL, and the net PnL of TP/SL exits on random entries
3. How much **volatility at entry** and **time of day** drive MFE. This matters because a signal that fires in volatile moments gets bigger MFE *and* bigger MAE for free.
4. A matched comparison for your real trade log: same side, same duration, same month, same volatility decile.

**Conventions:** entry = close of the entry bar; the path is bars t+1…t+H, using their high/low. If one bar touches both TP and SL, it's counted as SL (conservative). Costs: 20 bps round trip.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass
import excursions as ex

DATA_SOURCE = "binance"     # or "synthetic" (code check only)

START = pd.Timestamp("2024-12-30", tz="UTC")
END = pd.Timestamp("2026-01-02", tz="UTC")       # exclusive; 1 day of buffer for forward paths
EVAL_START = pd.Timestamp("2025-01-01", tz="UTC")
EVAL_END = pd.Timestamp("2026-01-01", tz="UTC")

HORIZONS_H = [0.5, 1, 2, 4, 8, 12, 24]   # intraday holding periods (hours)
THRESH = [0.005, 0.01, 0.02, 0.03]       # MFE thresholds
COST = 0.002                             # 20 bps round trip (0.1% taker each side)

# Your reported MFE hit rates (duration not specified) - drawn as markers on the survival plot
REPORTED = {0.01: 0.74, 0.02: 0.38, 0.03: 0.18}

TRADES_CSV = None   # e.g. "trades.csv" with columns entry_time, exit_time, side (+1/-1), UTC

In [ ]:
if DATA_SOURCE == "binance":
    from binance_5m import load_binance_5m
    bars = load_binance_5m(START, END)
else:
    import ss_bands as sb
    idx = pd.date_range(START, END - pd.Timedelta(minutes=5), freq="5min")
    bars, _ = sb.synthetic_5m(idx, trends=True, jumps=True)
    print("SYNTHETIC data - numbers say nothing about BTC.")
mask = (bars.index >= EVAL_START) & (bars.index < EVAL_END)
print(len(bars), "bars;", mask.sum(), "entries evaluated")

## 1. Unconditional MFE / MAE, random entries over all of 2025

Read across a row: if you enter at a random 5m close and hold for `hold_h` hours, this is how often price reaches +X% at some point (`P(MFE>=X)`), plus the typical favourable and adverse excursions.

In [ ]:
tab_long = ex.mfe_table(bars, mask, HORIZONS_H, THRESH, side="long")
tab_short = ex.mfe_table(bars, mask, HORIZONS_H, THRESH, side="short")
print("LONG"); display(tab_long.round(3))
print("SHORT"); display(tab_short.round(3))

## 2. MFE survival curves vs your reported numbers

Each line is P(MFE ≥ x) for one holding period. The black markers are your reported 74% / 38% / 18%. **If your markers sit on or below the line for your typical holding time, your entries are not finding moves that random entries don't.** (Long side shown; the short side is in the table above.)

In [ ]:
grid = np.linspace(0, 0.04, 161)
cmap = plt.get_cmap("viridis")
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
for i, hh in enumerate(HORIZONS_H):
    e = ex.forward_excursions(bars, int(hh * 12))[mask]
    col = cmap(i / (len(HORIZONS_H) - 1))
    axes[0].plot(100 * grid, 100 * ex.survival(e.long_mfe, grid), lw=2, color=col, label=f"{hh:g} h")
    axes[1].plot(100 * grid, 100 * ex.survival(e.long_mae, grid), lw=2, color=col, label=f"{hh:g} h")
axes[0].plot([100 * k for k in REPORTED], [100 * v for v in REPORTED.values()], "o", ms=9,
             color="black", label="your trades")
axes[0].set_title("Random long entries: P(MFE ≥ x)"); axes[1].set_title("Random long entries: P(MAE ≥ x)")
for a in axes:
    a.set_xlabel("x (% from entry)"); a.grid(alpha=.2); a.set_xlim(0, 4)
axes[0].set_ylabel("share of entries (%)"); axes[0].legend(frameon=False, title="hold")
plt.tight_layout(); plt.show()

## 3. TP / SL first passage on random entries

This is the honest version of "74% reached +1%": what matters is whether +TP comes **before** −SL. On a fair market, gross ≈ 0 for every TP/SL pair; TP/SL choices only change the *shape* of outcomes (win rate vs win size), not the mean. Any cell clearly above 0 gross reflects the drift of 2025, not an exit skill.

In [ ]:
TPS = [0.005, 0.01, 0.015, 0.02, 0.03]
SLS = [0.005, 0.01, 0.015, 0.02, 0.03]
for H_h in (4, 12):
    for side, name in ((1, "LONG"), (-1, "SHORT")):
        g = ex.tp_sl_grid(bars, mask, H_h * 12, TPS, SLS, side=side, cost=COST)
        print(f"{name}, time exit {H_h} h — P(TP first)    |    net bps per trade")
        display(pd.concat([g.pivot(index="tp_pct", columns="sl_pct", values="p_tp").round(3),
                           g.pivot(index="tp_pct", columns="sl_pct", values="net_bps").round(1)],
                          axis=1, keys=["P(TP first)", "net bps"]))

## 4. Volatility at entry drives MFE

Entries are bucketed by EWMA (1-day) volatility decile at the entry bar. If P(MFE ≥ 1%) swings a lot across deciles, then any entry signal that tends to fire after big moves (most do) will show high MFE without any skill. That's why section 6 matches random entries by volatility decile.

In [ ]:
H_REF = 4  # hours
e = ex.forward_excursions(bars, H_REF * 12)
vol = ex.ewma_vol(bars["close"])
df = pd.DataFrame(dict(vol=vol, mfe=e.long_mfe, mae=e.long_mae))[mask].dropna()
df["decile"] = pd.qcut(df.vol, 10, labels=False) + 1
vt = df.groupby("decile").agg(vol_5m_pct=("vol", lambda v: 100 * v.median()),
                              p_mfe_1=("mfe", lambda m: (m >= .01).mean()),
                              p_mae_1=("mae", lambda m: (m >= .01).mean()),
                              med_mfe_pct=("mfe", lambda m: 100 * m.median()))
display(vt.round(3))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(vt.index, 100 * vt.p_mfe_1, color="#2a6fdb", width=.7)
ax.set_xlabel("EWMA volatility decile at entry (1 = calmest)"); ax.set_ylabel(f"P(MFE ≥ 1% within {H_REF} h), %")
ax.set_title("Random long entries: MFE depends heavily on volatility at entry"); ax.grid(alpha=.2, axis="y")
ax.set_xticks(vt.index); plt.tight_layout(); plt.show()

## 5. Time of day and month

Hour is the UTC hour of the entry close (PKT = UTC+5). The monthly table shows how much the 2025 regime alone moves the baseline. For example, a downtrending month makes random longs look bad and random shorts look good.

In [ ]:
df2 = pd.DataFrame(dict(mfe=e.long_mfe, mae=e.long_mae, ret_l=e.long_ret, ret_s=e.short_ret,
                        smfe=e.short_mfe))[mask].dropna()
close_hour = (df2.index + pd.Timedelta(minutes=5)).hour
hr = df2.groupby(close_hour).agg(p_mfe_1=("mfe", lambda m: (m >= .01).mean()),
                                 p_mae_1=("mae", lambda m: (m >= .01).mean()))
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(hr.index, 100 * hr.p_mfe_1, color="#2a6fdb", width=.7)
ax.set_xlabel("Entry hour (UTC)"); ax.set_ylabel(f"P(MFE ≥ 1% within {H_REF} h), %")
ax.set_title("Random long entries by hour of day"); ax.set_xticks(range(24)); ax.grid(alpha=.2, axis="y")
plt.tight_layout(); plt.show()

mo = df2.groupby(df2.index.tz_convert(None).to_period("M")).agg(
    long_p_mfe_1=("mfe", lambda m: (m >= .01).mean()), short_p_mfe_1=("smfe", lambda m: (m >= .01).mean()),
    long_mean_ret_bps=("ret_l", lambda r: 1e4 * r.mean()), short_mean_ret_bps=("ret_s", lambda r: 1e4 * r.mean()))
print(f"By month, {H_REF} h hold"); display(mo.round(3))

## 6. Your trades vs matched random entries

For each real trade, 200 random entries are drawn with the **same side, same duration, within ±15 days, and in the same volatility decile**. The bootstrap builds random portfolios (one match per real trade) to give a 5–95% band and a one-sided p-value: the share of random portfolios that did at least as well.

- Entry and exit are priced at the close of the last candle completed at `entry_time` / `exit_time`.
- On synthetic tests the p-values are roughly uniform but run slightly hot, so **treat p < 0.01 as the bar**, not 0.05.
- Read MFE and MAE together: a higher MFE with an equally higher MAE is just volatility, not skill.

In [ ]:
if TRADES_CSV is None:
    print("Set TRADES_CSV to your trade log to run this section.")
else:
    trades = pd.read_csv(TRADES_CSV)
    trades["entry_time"] = pd.to_datetime(trades["entry_time"], utc=True)
    trades["exit_time"] = pd.to_datetime(trades["exit_time"], utc=True)
    real = ex.trade_excursions(bars, trades)
    print(len(real), "trades; median duration", real.D.median() / 12, "h")
    rand = ex.matched_random(bars, real, n_per=200, window_days=15, vol_match=True)
    display(ex.compare_real_vs_random(real, rand, THRESH).round(4))
    # exit efficiency: how much of the MFE each trade kept
    real["kept_of_mfe"] = real.ret / real.mfe.where(real.mfe > 0)
    print("median share of MFE kept at exit:", round(real.kept_of_mfe.median(), 3))

## 7. Reading the results

- **Section 2 first:** find the curve for your typical holding time. If your 74% / 38% / 18% markers sit on or below it, the MFE you saw is ordinary BTC volatility, and the exit wasn't throwing away an edge.
- **Section 3:** every random-entry TP/SL cell should be about −20 bps net. That's the bar an entry signal must clear. Exit rules alone can't lift it.
- **Sections 4–5:** if volatility deciles or hours move P(MFE ≥ 1%) by 2–3×, any unmatched comparison of your trades is meaningless. Use section 6.
- **Section 6:** an entry edge shows up as higher MFE **without** higher MAE, or a better mean return, with p < 0.01. Only then is it worth working on the exit (CUSUM / trailing stop, next notebook).